In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("C:/Users/hp/Downloads/a/05. Database RP May 2025 - AC REGISTER.xlsx", sheet_name="Raw", skiprows=1)

In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_percentage_error

In [4]:
print(df[['COCKPIT CREW PERSON', 'CABIN CREW PERSON']].head())

   COCKPIT CREW PERSON  CABIN CREW PERSON
0           299.002595          46.533742
1           388.627069          46.533742
2           279.797350          46.533742
3           331.011335          46.533742
4           292.600847          46.533742


In [5]:
df = df.iloc[:, 1:]

In [6]:
print([df])

[        YEAR PERIODE AIRCRAFT TYPE    FLIGHT NUMBER_CITYPAIR SERVICE TYPE  \
0       2024     JAN         AC738  GA0072.CGK-TKG.[CGK-TKG]          DOM   
1       2024     JAN         AC738  GA0072.CGK-TKG.[CGK-TKG]          DOM   
2       2024     JAN         AC738  GA0072.CGK-TKG.[CGK-TKG]          DOM   
3       2024     JAN         AC738  GA0072.CGK-TKG.[CGK-TKG]          DOM   
4       2024     JAN         AC738  GA0072.CGK-TKG.[CGK-TKG]          DOM   
...      ...     ...           ...                       ...          ...   
110775  2025     MAY         AC738  GA6054.KDI-UPG.[KDI-UPG]          DOM   
110776  2025     MAY         AC73H  GA2084.CGK-YIA.[CGK-YIA]          DOM   
110777  2025     MAY         AC73H  GA2094.YIA-CGK.[YIA-CGK]          DOM   
110778  2025     MAY         AC73H  GA4324.CGK-AMI.[CGK-AMI]          DOM   
110779  2025     MAY         AC73H  GA4344.AMI-CGK.[AMI-CGK]          DOM   

        SUB-SERVICE ROUNDTRIPROUTE FLIGHT ROUTE AC REG      DATE  ...  \
0

In [7]:
df1 = df[
    (df['BLOCK HOURS'] > 0) & 
    (df['ASK (000)'] > 0)
].copy()

In [8]:
df_num = df.select_dtypes(include=["number"])

corr = df_num.corrwith(df_num['COCKPIT CREW PERSON']).sort_values(ascending=False).dropna()

print(corr.head(30))

c:\Users\hp\Downloads\acopy\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\Downloads\acopy\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


COCKPIT CREW PERSON                  1.000000
TOTAL INDIRECT COSTS                 0.990191
BLOCK HOURS                          0.956423
FLIGHT HOURS                         0.955747
LEASE AIRCRAFT                       0.954247
FLIGHT KILOMETERS                    0.952442
TOTAL DIRECT,INDIRECT,FLEET COSTS    0.950749
TOTAL BO COSTS                       0.950725
TOTAL DIRECT AND INDIRECT COSTS      0.948206
TOTAL COSTS                          0.946607
TOTAL DIRECT FLIGHT COSTS            0.945756
CABIN CREW PERSON                    0.945605
TOTAL DIRECT COSTS                   0.940298
TOTAL FLEET COST                     0.937117
FUEL AIRCRAFT                        0.931193
FUEL BURN (IN LITER)                 0.927840
ASK (000)                            0.925489
ATK PASSENGER (000)                  0.925180
ASK (000) C CLASS                    0.924679
ATK (000)                            0.922127
ASK (000) Y CLASS                    0.918426
CABIN CREW TRAVEL                 

In [9]:
df_num = df.select_dtypes(include=["number"])

corr = df_num.corrwith(df_num['CABIN CREW PERSON']).sort_values(ascending=False).dropna()

print(corr.head(30))

c:\Users\hp\Downloads\acopy\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\hp\Downloads\acopy\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


CABIN CREW PERSON                    1.000000
TOTAL INDIRECT COSTS                 0.979795
ASK (000)                            0.975095
ATK PASSENGER (000)                  0.975065
ASK (000) Y CLASS                    0.973846
ATK (000)                            0.969257
FUEL BURN (IN LITER)                 0.961062
TOTAL DIRECT,INDIRECT,FLEET COSTS    0.958635
TOTAL COSTS                          0.957332
TOTAL BO COSTS                       0.956634
TOTAL DIRECT FLIGHT COSTS            0.954497
TOTAL DIRECT AND INDIRECT COSTS      0.954451
CABIN CREW TRAVEL                    0.953204
FUEL AIRCRAFT                        0.952853
LEASE AIRCRAFT                       0.951243
TOTAL FLEET COST                     0.950895
TOTAL DIRECT COSTS                   0.948150
COCKPIT CREW PERSON                  0.945605
FLIGHT KILOMETERS                    0.941807
FLIGHT HOURS                         0.929055
MAINTENANCE RESERVE                  0.928159
BLOCK HOURS                       

In [10]:
df_group = df1.groupby(["AC REG", "PERIODE"]).agg({
    "COCKPIT CREW PERSON": "sum",
    "CABIN CREW PERSON": "sum",

    "BLOCK HOURS": "sum",
    "FLIGHT HOURS": "sum",
    "FLIGHT KILOMETERS": "sum",

    "ASK (000) Y CLASS": "sum",
    "ASK (000) C CLASS": "sum",

    "LEASE AIRCRAFT": "mean",
    "FUEL BURN (IN LITER)": "sum",
    "NUMBER OF LANDING": "sum",

    "AIRCRAFT TYPE": "first", 
    "SERVICE TYPE": "first",
}).reset_index()

In [11]:
fitur_cockpit = [
    'BLOCK HOURS',  
    'FLIGHT HOURS',        
    'FLIGHT KILOMETERS',    
    'NUMBER OF LANDING', 
    'LEASE AIRCRAFT',   
    'AIRCRAFT TYPE',
    'SERVICE TYPE',
    'PERIODE',
    'AC REG'
]

In [12]:
fitur_cabin = [
    'ASK (000) Y CLASS',  
    'ASK (000) C CLASS',
    "BLOCK HOURS",
    "FUEL BURN (IN LITER)",
    'LEASE AIRCRAFT',      
    'AIRCRAFT TYPE',
    'NUMBER OF LANDING',
    'SERVICE TYPE',
    'PERIODE',
    'AC REG'
]

In [13]:
def train_log_model(target_name, features_list, df_data):
    print(f"\n{'='*10} TRAINING LOG-MODEL: {target_name} {'='*10}")
    
    data_train = df_data[df_data[target_name] > 0].copy()
    
    X = data_train[features_list]
    y = data_train[target_name]
    
    splitter = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups=X["AC REG"]))
    
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    y_train_log = np.log1p(y_train) 
    
    cat_cols = ['AIRCRAFT TYPE', 'SERVICE TYPE', 'PERIODE']
    num_cols = list(set(features_list) - set(cat_cols) - {'AC REG'})
    
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoder.fit(X_train[cat_cols])
    
    X_train_enc = pd.DataFrame(encoder.transform(X_train[cat_cols]), columns=encoder.get_feature_names_out(cat_cols), index=X_train.index)
    X_test_enc = pd.DataFrame(encoder.transform(X_test[cat_cols]), columns=encoder.get_feature_names_out(cat_cols), index=X_test.index)
    
    X_train_final = pd.concat([X_train[num_cols], X_train_enc], axis=1)
    X_test_final = pd.concat([X_test[num_cols], X_test_enc], axis=1)
    
    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        objective="reg:squarederror"
    )
    
    model.fit(X_train_final, y_train_log)
    
    y_pred_log = model.predict(X_test_final)
    y_pred_real = np.expm1(y_pred_log)
    
    mape = mean_absolute_percentage_error(y_test, y_pred_real)
    print(f"MAPE Score: {mape:.4f} ({mape*100:.2f}%)")
    
    return model, y_test, y_pred_real


bst_cb, y_test_cb, y_pred_cb = train_log_model('CABIN CREW PERSON', fitur_cabin, df_group)

bst_cp, y_test_cp, y_pred_cp = train_log_model('COCKPIT CREW PERSON', fitur_cockpit, df_group)

print("\n--- Contoh Hasil Prediksi Cockpit ---")
res = pd.DataFrame({'Actual': y_test_cp, 'Predicted': y_pred_cp})
res['Diff %'] = abs(res['Actual'] - res['Predicted']) / res['Actual'] * 100
print(res.head())


========== TRAINING LOG-MODEL: CABIN CREW PERSON ==========


MAPE Score: 0.0452 (4.52%)

========== TRAINING LOG-MODEL: COCKPIT CREW PERSON ==========
MAPE Score: 0.0462 (4.62%)

--- Contoh Hasil Prediksi Cockpit ---
          Actual      Predicted    Diff %
0  181281.594945  184895.250000  1.993393
1   85733.032242   87926.406250  2.558377
2  105891.273405  104483.218750  1.329717
3  216285.904694  213924.093750  1.091986
4  226064.265634  226012.515625  0.022892
